In [1]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    # Colab already ships torch, numpy, scipy, sklearn, matplotlib, seaborn, tqdm, requests
    # Install only what's missing
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mne>=1.6', 'moabb>=1.0', 'xgboost'])
    print('Colab environment ready.')
else:
    print('Local environment — ensure you ran: pip install -e .')

Colab environment ready.


In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from physreve import (
    PhysREVEConfig,
    LabeledEEGDataset, make_split_loaders,
    run_baseline_finetune, run_mae_pretraining, run_pretraining, run_finetuning,
    extract_features, run_ml_baselines,
    compare_models,
)
from physreve.physics import build_leadfield, motor_roi_indices

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
# Core
import numpy as np
import torch
import torch.nn as nn

# Analysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# RSA
from scipy.spatial.distance import pdist, squareform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Build leadfield + instantiate PhysREVE model ─────────────────────────────
L_col_np, L_row_np, src_pos, info, L_raw = build_leadfield(
    CH_NAMES, sfreq=SFREQ, verbose=False, return_raw=True
)

L_col = torch.tensor(L_col_np, dtype=torch.float32).to(device)  # (C, N_src)
L_row = torch.tensor(L_row_np, dtype=torch.float32).to(device)  # (C, N_src)

N_CH = len(CH_NAMES)
elec_xyz_np = np.array([info["chs"][i]["loc"][:3] for i in range(N_CH)])
elec_xyz    = torch.tensor(elec_xyz_np, dtype=torch.float32).to(device)  # (C, 3)

# patch_size=32 divides 256 samples evenly → 8 patches
PATCH_SIZE = 32
cfg = PhysREVEConfig(
    d_model=256, n_heads=8, n_layers=6,
    patch_size=PATCH_SIZE,
    n_sources=L_col.shape[1],
    dropout=0.1,
)

lf_bias = LeadfieldAttentionBias(L_row, cfg).to(device)
model   = PhysREVEPretrainModel(cfg, lf_bias).to(device)
model.eval()
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params  |  device: {device}")
print(f"Channels: {N_CH}  |  Sources: {L_col.shape[1]}  |  Patch size: {PATCH_SIZE}")

In [3]:
# ADD: synthetic dataset generator
def generate_sine_wave(freq, fs=256, duration=1.0, noise_std=0.1):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    signal = np.sin(2 * np.pi * freq * t)
    noise = np.random.normal(0, noise_std, size=signal.shape)
    return signal + noise

In [ ]:
def generate_burst(freq, fs=256, duration=1.0, noise_std=0.1):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    signal = np.zeros_like(t)

    mask = (t > 0.3) & (t < 0.6)
    signal[mask] = np.sin(2 * np.pi * freq * t[mask])

    noise = np.random.normal(0, noise_std, size=signal.shape)
    return signal + noise

In [4]:
# ── Synthetic dataset: sine waves broadcast across all EEG channels ────────
FS    = 256   # samples per second
freqs = [4, 8, 10, 20, 40]
X, y  = [], []

for f in freqs:
    for _ in range(200):
        sig   = generate_sine_wave(f, fs=FS)          # (256,)
        # Broadcast to all EEG channels with small per-channel noise
        trial = np.stack([
            sig + np.random.normal(0, 0.05, sig.shape)
            for _ in range(N_CH)
        ], axis=0)                                      # (C, 256)
        X.append(trial)
        y.append(f)

X = np.array(X, dtype=np.float32)  # (1000, C, 256)
y = np.array(y)
print(f"Dataset: {X.shape}  (samples, channels={N_CH}, time={FS})")
print(f"Classes: {freqs}")

In [5]:
# ── Encode trials with PhysREVE — extract CLS token embeddings ──────────────
BATCH = 64
elec_xyz_1 = elec_xyz.unsqueeze(0)  # (1, C, 3) — broadcast over batch

Z_list = []
model.eval()
with torch.no_grad():
    for start in range(0, len(X), BATCH):
        Xb  = torch.tensor(X[start:start+BATCH]).to(device)   # (B, C, T)
        B   = Xb.shape[0]
        xyz = elec_xyz_1.expand(B, -1, -1)                    # (B, C, 3)
        cls_out, _, _, _ = model.encoder(Xb, xyz)             # (B, d_model)
        Z_list.append(cls_out.cpu())

Z = torch.cat(Z_list, dim=0).numpy()  # (N_trials, d_model)
print(f"Embeddings: {Z.shape}  (trials × d_model)")